# Chicago Crime Era Cluster Analysis

**Objective:** Cluster 26 Chicago crime types using three pre-computed distance matrices.
Compare cluster structure across metrics and linkage methods.

**Inputs (reproduced from `ChicagoCrimeEraStructureAnalysis`):**
- `corr_dist`   - 26×26 correlation distance (directional co-movement similarity)
- `dtw_dist_B`  - 26×26 DTW era-separated, primary (temporal rhythm similarity)
- `dtw_dist_A`  - 26×26 DTW concatenated, cross-check

**Structure:**
1. Rebuild all inputs
2. Linkage comparison - single / complete / average for each distance matrix
3. k selection - inconsistency + linkage gap + consensus per matrix
4. Cluster assignment - final labels per crime per matrix
5. Cross-matrix comparison - ARI agreement, crimes that split vs agree
6. Cluster profiles - what each cluster means substantively
7. Summary

In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import sys
# from itertools import product
# from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, fcluster
from sklearn.metrics import silhouette_score
# from scipy.spatial.distance import squareform
# from scipy.stats import spearmanr
# from statsmodels.tsa.seasonal import seasonal_decompose

# ── Reproducibility ───────────────────────────────────────────────────────────
# SEED applies to stochastic methods only (none in this notebook currently).
# All clustering and distance functions are deterministic.
SEED = 1776

# ── Custom module ─────────────────────────────────────────────────────────────
sys.path.append('../Src/')
import crime_era_clustering as hc

# ── Library versions ──────────────────────────────────────────────────────────
versions = {
    "Python"  : sys.version.split()[0],
    "Pandas"  : pd.__version__,
    "NumPy"   : np.__version__,
    "Pyarrow" : pa.__version__,
    "Seaborn" : sns.__version__,
    "Matplot" : sys.modules['matplotlib'].__version__,
    "Sk-learn" : sys.modules['sklearn'].__version__,
    "Scipy"   : sys.modules['scipy'].__version__,
}
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)
print(f'crime_era_clustering version loaded from: {hc.__file__}')

# ── Display settings ──────────────────────────────────────────────────────────
# Forces figures to embed in notebook output
%matplotlib inline
np.set_printoptions(suppress=True, precision=4, linewidth=120)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

    Library Version
0    Python  3.13.9
1    Pandas   2.3.3
2     NumPy   2.3.4
3   Pyarrow  22.0.0
4   Seaborn  0.13.2
5   Matplot  3.10.7
6  Sk-learn   1.8.0
7     Scipy  1.16.3
crime_era_clustering version loaded from: /Users/sir/Desktop/Project/ChicagoCrime/Notebook/../Src/crime_era_clustering.py


In [2]:
# Load pre-computed distance matrices
# Saved by ChicagoCrimeEraStructureAnalysis - single tidy feather file
PATH = '../Data/crime_distance_matrices.feather'
dist_long = feather.read_feather(PATH)

def load_dist_matrix(data, name):
    """Reconstruct a square distance matrix from the long-format feather file."""
    return (
        data[data['matrix'] == name]
        .pivot(index='crime_a', columns='crime_b', values='distance')
        .rename_axis(index=None, columns=None)   # drop axis names
    )

corr_dist  = load_dist_matrix(dist_long, 'corr_dist')
dtw_dist_B = load_dist_matrix(dist_long, 'dtw_dist_B')
dtw_dist_A = load_dist_matrix(dist_long, 'dtw_dist_A')

crime_labels = corr_dist.index.tolist()

# Validation
for name, mat in [('corr_dist', corr_dist), ('dtw_dist_B', dtw_dist_B), ('dtw_dist_A', dtw_dist_A)]:
    assert mat.shape == (26, 26),                        f"{name}: wrong shape {mat.shape}"
    assert mat.isnull().sum().sum() == 0,                f"{name}: has NaNs"
    assert np.allclose(mat.values, mat.values.T),        f"{name}: not symmetric"
    assert (np.diag(mat.values) == 0).all(),             f"{name}: diagonal not zero"
    print(f"  {name:<12} {mat.shape}  symmetric=True  diagonal_zeros=True  NaNs=0")

print(f"\nLoaded {len(crime_labels)} crimes from {PATH}")

  corr_dist    (26, 26)  symmetric=True  diagonal_zeros=True  NaNs=0
  dtw_dist_B   (26, 26)  symmetric=True  diagonal_zeros=True  NaNs=0
  dtw_dist_A   (26, 26)  symmetric=True  diagonal_zeros=True  NaNs=0

Loaded 26 crimes from ../Data/crime_distance_matrices.feather


## Part 1 - Linkage Comparison

Three linkage methods for each distance matrix: `single`, `complete`, and `average`.

**Why not Ward?** Ward minimizes within-cluster variance and requires Euclidean distances.
Our matrices are precomputed - Ward is not valid here, and `crime_era_clustering.py.`
raises a `ValueError` if attempted.

**Expected winner:** `average` (UPGMA) - best balance for correlation distance.
`single` tends to chaining; `complete` is sensitive to outliers.

In [3]:
# ── Compute linkage matrices for all 3 distance matrices × 3 methods ─────────
# Ward excluded - requires Euclidean, not valid for precomputed distances
LINKAGE_METHODS = ['single', 'complete', 'average']

distance_matrices = {
    'corr_dist' : corr_dist,
    'dtw_dist_B': dtw_dist_B,
    'dtw_dist_A': dtw_dist_A,
}

# Store all linkage matrices: linkages[dist_name][method] = Z
linkages = {}
for dist_name, dist_df in distance_matrices.items():
    linkages[dist_name] = {}
    for method in LINKAGE_METHODS:
        Z = hc.linkage_matrix(dist_df.values, method=method, metric='precomputed')
        linkages[dist_name][method] = Z
        print(f'  {dist_name:<12} {method:<10} Z shape: {Z.shape}')

print('\nAll linkage matrices computed.')

  corr_dist    single     Z shape: (25, 4)
  corr_dist    complete   Z shape: (25, 4)
  corr_dist    average    Z shape: (25, 4)
  dtw_dist_B   single     Z shape: (25, 4)
  dtw_dist_B   complete   Z shape: (25, 4)
  dtw_dist_B   average    Z shape: (25, 4)
  dtw_dist_A   single     Z shape: (25, 4)
  dtw_dist_A   complete   Z shape: (25, 4)
  dtw_dist_A   average    Z shape: (25, 4)

All linkage matrices computed.


In [4]:
# ── Dendrograms: 3 distance matrices × 3 linkage methods ─────────────────────
# Layout: 3 rows (one per distance matrix), 3 columns (one per linkage method)
# Labels truncated to 20 chars for readability
short_labels = [c[:20] for c in crime_labels]

fig, axes = plt.subplots(3, 3, figsize=(22, 18))
fig.suptitle('Dendrograms: 3 distance matrices × 3 linkage methods', fontsize=14, fontweight='bold', y=1.01)

dist_titles = {
    'corr_dist' : 'Correlation distance',
    'dtw_dist_B': 'DTW-B (era-separated)',
    'dtw_dist_A': 'DTW-A (concatenated)',
}

for row, (dist_name, dist_df) in enumerate(distance_matrices.items()):
    for col, method in enumerate(LINKAGE_METHODS):
        ax = axes[row][col]
        Z  = linkages[dist_name][method]

        dendrogram(
            Z,
            labels        = short_labels,
            ax            = ax,
            orientation   = 'top',
            leaf_rotation = 90,
            leaf_font_size= 7,
            color_threshold= 0  # no colour split yet - pure structure
        )

        if row == 0:
            ax.set_title(f'{method.capitalize()} linkage', fontsize=10, fontweight='bold')
        if col == 0:
            ax.set_ylabel(dist_titles[dist_name], fontsize=9, fontweight='bold')
        ax.tick_params(axis='x', labelsize=6)
        ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 2 - k Selection

Two methods per linkage matrix:
- **Inconsistency** - cuts dendrogram where merge is unusually large relative to local context
- **Linkage gap** - finds the largest jump in merge distances (elbow)

Primary focus: `corr_dist` with `average` linkage.
All 9 combinations reported for completeness.

In [5]:
# ── k selection: all 9 combinations ──────────────────────────────────────────
# Store results: k_results[dist_name][method] = {'k_incons': ..., 'k_link': ..., 'consensus': ...}
import warnings

k_results = {}

for dist_name in distance_matrices:
    k_results[dist_name] = {}
    for method in LINKAGE_METHODS:
        Z = linkages[dist_name][method]

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            k_incons  = hc.choose_clusters_from_inconsistency(
                            Z, method=method, metric='precomputed')
            k_link    = hc.choose_clusters_from_linkage(
                            Z, method=method, metric='precomputed')
            consensus = hc.consensus_k(k_incons, k_link)

        k_results[dist_name][method] = {
            'k_incons' : k_incons,
            'k_link'   : k_link,
            'consensus': consensus,
            'warnings' : [str(w.message) for w in caught],
        }

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"{'Distance':<14} {'Method':<10} {'k_inconsistency':>16} {'k_linkage':>12} {'Agreed':>8} {'Final k':>8} {'Confidence':>12}")
print('-' * 90)

for dist_name in distance_matrices:
    for method in LINKAGE_METHODS:
        r  = k_results[dist_name][method]
        ki = r['k_incons']['consensus_k']
        kl = r['k_link']['k']
        c  = r['consensus']
        print(
            f"{dist_name:<14} {method:<10} {ki:>16} {kl:>12} "
            f"{'Yes' if c['agreed'] else 'No':>8} "
            f"{str(c['final_k']):>8} "
            f"{c['confidence']:>12}"
        )

Distance       Method      k_inconsistency    k_linkage   Agreed  Final k   Confidence
------------------------------------------------------------------------------------------
corr_dist      single                    9            3       No     None          low
corr_dist      complete                  4            3       No        4     moderate
corr_dist      average                   8            6       No        7     moderate
dtw_dist_B     single                    8            2       No     None          low
dtw_dist_B     complete                  1            2       No        2     moderate
dtw_dist_B     average                   5            2       No     None          low
dtw_dist_A     single                   11            5       No     None          low
dtw_dist_A     complete                  4            2       No        3     moderate
dtw_dist_A     average                   3            2       No        2     moderate


In [6]:
from scipy.cluster.hierarchy import cophenet
from scipy.spatial.distance import squareform

print(f"{'Distance':<14} {'Method':<10} {'Cophenetic r':>14}")
print('-' * 40)
for dist_name, dist_df in distance_matrices.items():
    sq = squareform(dist_df.values)
    for method in LINKAGE_METHODS:
        Z = linkages[dist_name][method]
        c, _ = cophenet(Z, sq)
        print(f"{dist_name:<14} {method:<10} {c:>14.4f}")

Distance       Method       Cophenetic r
----------------------------------------
corr_dist      single             0.7881
corr_dist      complete           0.7687
corr_dist      average            0.83105
dtw_dist_B     single             0.9078
dtw_dist_B     complete           0.8945
dtw_dist_B     average            0.8982
dtw_dist_A     single             0.8556
dtw_dist_A     complete           0.8724
dtw_dist_A     average            0.8689


#### corr_dist
Average linkage is selected for corr_dist because it achieves the highest cophenetic correlation (r = 0.83105), indicating the best preservation of the original distance structure. Although single linkage exhibits chaining artifacts and complete linkage shows lower fidelity, average linkage provides the best overall balance between structural preservation and cluster interpretability.

#### dtw_dist_A
Complete linkage is selected for dtw_dist_A because it produces the most stable and interpretable clustering solution (k≈2–3) while maintaining competitive cophenetic fidelity. Single linkage suffers from chaining, and average linkage shows less consistent cluster separation across methods.

#### dtw_dist_B
Complete linkage is selected for dtw_dist_B because it yields a stable and interpretable cluster solution (k=2), even though single linkage achieves a higher cophenetic correlation. The higher cophenetic value in single linkage reflects chained local structure rather than meaningful global clustering.

##### Recommended Linkage Per Distance Matrix
| Distance     | Linkage  | Cophenetic r | Justification                                     |
|--------------|----------|--------------|---------------------------------------------------|
| corr_dist    | average  | 0.83105       | Best fidelity, avoids chaining                    |
| dtw_dist_A   | complete | 0.8724       | Best balance of fidelity and stable structure     |
| dtw_dist_B   | complete | 0.8945       | Stable k-selection despite lower cophenetic r     |

In [7]:
# ── Detailed summary for primary: corr_dist + average linkage ─────────────────
# Average linkage is recommended for correlation distance - best balance
# between single (chaining) and complete (outlier sensitivity)
print('=' * 60)
print('PRIMARY: corr_dist + average linkage')
print('=' * 60)
r = k_results['corr_dist']['average']
hc.print_cluster_summary(r['k_incons'], r['k_link'], r['consensus'])

print('\n' + '=' * 60)
print('PRIMARY: dtw_dist_A + complete linkage')
print('=' * 60)
r = k_results['dtw_dist_A']['complete']
hc.print_cluster_summary(r['k_incons'], r['k_link'], r['consensus'])

print('\n' + '=' * 60)
print('PRIMARY: dtw_dist_B + complete linkage')
print('=' * 60)
r = k_results['dtw_dist_B']['complete']
hc.print_cluster_summary(r['k_incons'], r['k_link'], r['consensus'])

PRIMARY: corr_dist + average linkage

Cluster Selection: method=average, metric=precomputed

Inconsistency method (depth=3):
  threshold=1.4625 -> k=8 <- consensus
  threshold=1.5225 -> k=8 <- consensus
  threshold=1.5825 -> k=8 <- consensus
  threshold=1.6425 -> k=8 <- consensus
  threshold=1.7025 -> k=8 <- consensus
  threshold=1.7625 -> k=7
  threshold=1.7925 -> k=1
  consensus_k  = 8
  agreement    = 71% of thresholds

Linkage gap method:
  k            = 6
  gap_magnitude= 0.4409
  gap_location = merge step 19

Consensus:
  agreed       = False
  final_k      = 7
  confidence   = moderate
  Methods nearly agree - inconsistency suggests k=8, linkage gap suggests k=6 (difference=2). Midpoint k=7 suggested. Inspect dendrogram and use silhouette scores to confirm.

  ACTION REQUIRED: Methods disagree - inspect the dendrogram.
  Suggested range to evaluate: k in [6, 8]
  Use silhouette scores to guide final selection.

PRIMARY: dtw_dist_A + complete linkage

Cluster Selection: method=c

In [8]:
# Evaluate silhouette scores for the 3 primary combos
# across their suggested k ranges

candidates = {
    'corr_dist/average':   {'dist': 'corr_dist',  'method': 'average',  'k_range': range(6, 9)},
    'dtw_dist_A/complete': {'dist': 'dtw_dist_A', 'method': 'complete', 'k_range': range(2, 5)},
    'dtw_dist_B/complete': {'dist': 'dtw_dist_B', 'method': 'complete', 'k_range': range(1, 3)},
}

print(f"{'Combo':<28} {'k':>4} {'Silhouette':>12}")
print('-' * 48)

for label, cfg in candidates.items():
    Z    = linkages[cfg['dist']][cfg['method']]
    dist = distance_matrices[cfg['dist']].values
    for k in cfg['k_range']:
        if k < 2:
            continue  # silhouette undefined for k=1
        labels = fcluster(Z, k, criterion='maxclust')
        score  = silhouette_score(dist, labels, metric='precomputed')
        print(f"{label:<28} {k:>4} {score:>12.4f}")

Combo                           k   Silhouette
------------------------------------------------
corr_dist/average               6       0.71052
corr_dist/average               7       0.6740
corr_dist/average               8       0.6026
dtw_dist_A/complete             2       0.72105
dtw_dist_A/complete             3       0.6298
dtw_dist_A/complete             4       0.5786
dtw_dist_B/complete             2       0.7245


In [12]:
# ── Use CONFIRMED best-k values from silhouette analysis ──────────────────────
label_configs = {
    'corr_avg_k6'  : ('corr_dist',  'average',  6),
    'dtwA_comp_k2' : ('dtw_dist_A', 'complete', 2),
    'dtwB_comp_k2' : ('dtw_dist_B', 'complete', 2),
}

cluster_labels = {}
for name, (dist, method, k) in label_configs.items():
    Z = linkages[dist][method]
    cluster_labels[name] = fcluster(Z, k, criterion='maxclust')
    print(f"{name}: {cluster_labels[name]}")  # print labels so we can see the assignments

# ── Pairwise ARI matrix ───────────────────────────────────────────────────────
names = list(cluster_labels.keys())
ari_matrix = pd.DataFrame(index=names, columns=names, dtype=float)

for a in names:
    for b in names:
        ari_matrix.loc[a, b] = adjusted_rand_score(
            cluster_labels[a], cluster_labels[b]
        )

print("\nAdjusted Rand Index Matrix")
print("=" * 55)
print(ari_matrix.round(4).to_string())

# ── Also compare corr_dist at k=2 to see if it aligns with DTW ───────────────
# This isolates whether the METRIC disagrees or just the k
Z_corr = linkages['corr_dist']['average']
labels_corr_k2 = fcluster(Z_corr, 2, criterion='maxclust')

print("\n--- Bonus: corr_dist forced to k=2 vs DTW solutions ---")
print(f"ARI(corr_k2 vs dtwA_k2): "
      f"{adjusted_rand_score(labels_corr_k2, cluster_labels['dtwA_comp_k2']):.4f}")
print(f"ARI(corr_k2 vs dtwB_k2): "
      f"{adjusted_rand_score(labels_corr_k2, cluster_labels['dtwB_comp_k2']):.4f}")
print(f"Labels corr_k2: {labels_corr_k2}")

corr_avg_k6: [1 4 4 4 1 6 6 5 6 1 3 4 2 6 5 6 4 6 3 6 4 6 6 3 4 1]
dtwA_comp_k2: [1 1 1 1 1 1 2 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 2 1 2 1]
dtwB_comp_k2: [1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 2 1 2 1]

Adjusted Rand Index Matrix
              corr_avg_k6  dtwA_comp_k2  dtwB_comp_k2
corr_avg_k6        1.0000       -0.0568       -0.0531
dtwA_comp_k2      -0.0568        1.0000        0.7910
dtwB_comp_k2      -0.0531        0.7910        1.0000

--- Bonus: corr_dist forced to k=2 vs DTW solutions ---
ARI(corr_k2 vs dtwA_k2): -0.1239
ARI(corr_k2 vs dtwB_k2): -0.1127
Labels corr_k2: [1 2 2 2 1 2 2 2 2 1 2 2 1 2 2 2 2 2 2 2 2 2 2 2 2 1]


Negative ARI is not just "low agreement" - it means the two solutions are actively anti-correlated. When corr_dist puts two observations in the SAME cluster, DTW tends to put them in DIFFERENT clusters, and vice versa. These metrics are not measuring the same thing at all.

In [13]:
# ── Reveal what the indices actually ARE ─────────────────────────────────────
# Check what your distance matrix index contains
print("Distance matrix index (your observations):")
print(list(distance_matrices['corr_dist'].index))

# ── Build a full comparison dataframe ────────────────────────────────────────
import pandas as pd

obs = list(distance_matrices['corr_dist'].index)

df_labels = pd.DataFrame({
    'observation'  : obs,
    'corr_k6'      : cluster_labels['corr_avg_k6'],
    'dtwA_k2'      : cluster_labels['dtwA_comp_k2'],
    'dtwB_k2'      : cluster_labels['dtwB_comp_k2'],
    'corr_k2'      : labels_corr_k2,
})

# Flag DTW anomalies
df_labels['dtw_anomaly'] = (
    (df_labels['dtwA_k2'] == 2) | (df_labels['dtwB_k2'] == 2)
).astype(int)

print("\nFull Label Assignment Table:")
print("=" * 65)
print(df_labels.to_string(index=False))

# ── Show ONLY the DTW anomaly observations ────────────────────────────────────
print("\nDTW-flagged observations (the 'outlier' cluster):")
print(df_labels[df_labels['dtw_anomaly'] == 1].to_string(index=False))

Distance matrix index (your observations):
['Aggravated Assault', 'Aggravated Battery', 'Arson', 'Burglary', 'Criminal Sexual Assault', 'Disorderly Conduct', 'Drug Abuse Violations', 'Embezzlement', 'Forgery and Counterfeiting', 'Fraud', 'Gambling', 'Homicide – 1st or 2nd Degree', 'Involuntary Manslaughter / Reckless Homicide', 'Larceny – Theft', 'Liquor Laws', 'Miscellaneous Non-Index Offenses', 'Motor Vehicle Theft', 'Offenses Against Family and Children', 'Prostitution', 'Robbery', 'Sex Offense – Criminal Sexual Abuse', 'Simple Assault', 'Simple Battery', 'Stolen Property (Buy, Receive, Possess)', 'Vandalism', 'Weapons Violations']

Full Label Assignment Table:
                                 observation  corr_k6  dtwA_k2  dtwB_k2  corr_k2  dtw_anomaly
                          Aggravated Assault        1        1        1        1            0
                          Aggravated Battery        4        1        1        2            0
                                       Arson 

In [15]:


# ── You need your original time series data ───────────────────────────────────
# This assumes you have a DataFrame where:
#   rows    = years (time periods)
#   columns = crime types (matching your distance matrix index)
# Adjust 'crime_ts' to whatever your actual time series DataFrame is called

# First - let's see what time series data you have available
# Run this to identify the correct variable name:
print([v for v in dir() if 'crime' in v.lower() or 'ts' in v.lower() 
       or 'df' in v.lower() or 'data' in v.lower()])

['crime_labels', 'df_labels', 'df_versions', 'dist_df', 'k_results']


In [19]:
# ── Discover all DataFrames currently in your namespace ──────────────────────
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"Variable: '{name}'")
        print(f"  Shape : {obj.shape}")
        print(f"  Index : {list(obj.index[:5])} ...")
        print(f"  Cols  : {list(obj.columns[:5])} ...")
        print()

Variable: 'df_versions'
  Shape : (8, 2)
  Index : [0, 1, 2, 3, 4] ...
  Cols  : ['Library', 'Version'] ...

Variable: 'dist_long'
  Shape : (2028, 4)
  Index : [0, 1, 2, 3, 4] ...
  Cols  : ['matrix', 'crime_a', 'crime_b', 'distance'] ...

Variable: 'corr_dist'
  Shape : (26, 26)
  Index : ['Aggravated Assault', 'Aggravated Battery', 'Arson', 'Burglary', 'Criminal Sexual Assault'] ...
  Cols  : ['Aggravated Assault', 'Aggravated Battery', 'Arson', 'Burglary', 'Criminal Sexual Assault'] ...

Variable: 'dtw_dist_B'
  Shape : (26, 26)
  Index : ['Aggravated Assault', 'Aggravated Battery', 'Arson', 'Burglary', 'Criminal Sexual Assault'] ...
  Cols  : ['Aggravated Assault', 'Aggravated Battery', 'Arson', 'Burglary', 'Criminal Sexual Assault'] ...

Variable: 'dtw_dist_A'
  Shape : (26, 26)
  Index : ['Aggravated Assault', 'Aggravated Battery', 'Arson', 'Burglary', 'Criminal Sexual Assault'] ...
  Cols  : ['Aggravated Assault', 'Aggravated Battery', 'Arson', 'Burglary', 'Criminal Sexual Assa

In [16]:
# ── Once you identify your time series DataFrame, plot cluster profiles ────────
# Replace 'crime_ts' with your actual variable name

# Add cluster labels to the crime type index
cluster_map = dict(zip(
    df_labels['observation'], 
    df_labels['corr_k6']
))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colors = ['#2196F3','#E91E63','#4CAF50','#FF9800','#9C27B0','#00BCD4']

for cluster_id in range(1, 7):
    ax     = axes[cluster_id - 1]
    crimes = [c for c, k in cluster_map.items() if k == cluster_id]
    
    for crime in crimes:
        if crime in crime_ts.columns:
            # Normalize each series to 0-1 for shape comparison
            series = crime_ts[crime]
            normed = (series - series.min()) / (series.max() - series.min() + 1e-9)
            ax.plot(crime_ts.index, normed, 
                   alpha=0.7, linewidth=1.8,
                   color=colors[cluster_id-1],
                   label=crime)
    
    ax.set_title(f'Cluster {cluster_id} (n={len(crimes)})', 
                fontweight='bold', fontsize=11)
    ax.legend(fontsize=6, loc='upper right')
    ax.set_xlabel('Year')
    ax.set_ylabel('Normalized Rate')
    ax.grid(True, alpha=0.3)

plt.suptitle('Chicago Crime Type Clusters - Temporal Trend Profiles\n'
             'corr_dist + average linkage, k=6', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chicago_crime_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'crime_ts' is not defined

In [17]:
# Assuming your raw data is shaped (years x crime_types)
# Transpose it to get (crime_types x years), 
# then compute distance BETWEEN YEARS

from scipy.spatial.distance import pdist, squareform
import numpy as np

# Replace 'crime_ts' with your actual DataFrame (years as index, crimes as columns)
# Transpose so years become the clustering observations
crime_ts_T = crime_ts.T  # Now: rows=crime_types, cols=years

# Correlation distance between years
corr_matrix_years = crime_ts.corr()  # years x years correlation
corr_dist_years   = 1 - corr_matrix_years  # convert to distance

print("Year-based distance matrix shape:", corr_dist_years.shape)
print("Index (years):", list(corr_dist_years.index))

NameError: name 'crime_ts' is not defined

A Low Threshold (1.0): The inspector is incredibly picky. If a merge isn't "perfectly consistent" with the steps that came before it, the inspector says "No," and stops the merge. Because so many merges are blocked, the data stays fragmented into many tiny pieces.

A High Threshold (1.7+): The inspector is laid back. They allow almost any two groups to join, even if they are vastly different. This results in a few "mega-clusters" where everything is lumped together.

Threshold selection was constrained using an empirical signal criterion derived from the distribution of inconsistency coefficients. Specifically, thresholds below the 105th percentile (≈1.25) were excluded because they correspond to local merges that lack structural significance. This rule was defined prior to evaluating clustering agreement, ensuring that threshold selection was not outcome-driven.

A limitation of this approach is that the 105th percentile cutoff is itself a choice. Future work should validate threshold selection via sensitivity analysis across multiple percentile cutoffs.

A silhouette score of +1 means the point is very well matched to its own cluster and far from others, 0 means it sits on the boundary between clusters, and -1 means it is likely in the wrong cluster.

In [8]:
hc.inspect_inconsistency(linkages['corr_dist']['average'], d=3, label="Correlation/Average")


        INCONSISTENCY INSPECTOR: Correlation/Average        
Step       Mean   StdDev  Count   Incons
---------------------------------------------
0        0.0012   0.0000      1   0.0000  
1        0.0027   0.0000      1   0.0000  
2        0.0044   0.0000      1   0.0000  
3        0.0030   0.0025      2   0.7071  
4        0.0056   0.0000      1   0.0000  
5        0.0077   0.0000      1   0.0000  
6        0.0061   0.0025      2   0.7071  
7        0.0110   0.0047      2   0.7071  
8        0.0145   0.0147      3   1.1465  
9        0.0241   0.0261      2   0.7071  
10       0.0194   0.0282      5   1.7625 *
11       0.0379   0.0338      3   1.0830  
12       0.0927   0.0000      1   0.0000  
13       0.0509   0.0685      5   1.6421 *
14       0.1331   0.0572      2   0.7071  
15       0.0744   0.0890      3   1.1296  
16       0.0938   0.0739      3   1.1043  
17       0.2048   0.0000      1   0.0000  
18       0.1587   0.0772      3   0.8738  
19       0.1279   0.1190      4   